# Evolving convection of a Darcy fluid in a cross-bedded porous rectangle

$$
\mathbb{S}_{\psi,c}
\begin{cases}
\Omega = [0, \mathcal{A}X] \times [0, X] & \text{aspect ratio }\mathcal{A}=\mathcal{O}(1)\\
c_0(x,y)=\lim_{\epsilon\to0}\left(1+\text{erf}\left(\frac{y-Ra}{\epsilon Ra}\right)\right)+\mathcal{N}(x,y) & \text{perturbed diffusive base state} \\
c_{\text{D}}(x,y=X)=1 & \text{prescribed concentration on upper and lower boundaries} \\
c_{\text{N}}(x,y=0)=0 & \text{no-flux on lower, left and right boundaries}\\
c_{\text{N}}(x=0,y)=0 \\
c_{\text{N}}(x=\mathcal{A}X,y)=0 \\
\psi_{\text{D}}\vert_{\partial\Omega}=0 & \text{no-penetration on entire boundary}\\
\textbf{e}_g=-\textbf{e}_y & \text{vertically downward gravity}\\ 
\phi = 1 & \text{constant porosity} \\
\mathsf{D} = \mathsf{I} & \text{constant isotropic dispersion} \\ 
\mathsf{K} = \begin{pmatrix}
\cos^2\vartheta +\kappa\sin^2\vartheta & (1 -  \kappa)\cos\vartheta\sin\vartheta \\
(1 -  \kappa)\cos\vartheta\sin\vartheta & \cos^2\vartheta +\kappa\sin^2\vartheta\\
 \end{pmatrix} & \text{anisotropic permeability} \\ 
\mu = 1 & \text{constant viscosity} \\
\rho(c) = c & \text{linear density} 
\end{cases}
$$

In [ ]:
import numpy as np
from lucifex.fdm import AB2, CN
from lucifex.sim import run
from lucifex.utils.npy_utils import as_index
from lucifex.plt import plot_colormap, save_figure, create_animation, display_animation
from py.C14_darcy_evolving import darcy_convection_evolving_rectangle


vartheta = 45.0
simulation = darcy_convection_evolving_rectangle(
    aspect=2.0,
    Nx=64,
    Ny=64,
    cell='quadrilateral', 
    scaling='advective',
    Ra=500.0, 
    kappa=0.1,
    vartheta=vartheta,
    c_ampl=1e-3, 
    c_freq=(14, 14), 
    c_seed=(456, 987), 
    D_adv=AB2,
    D_diff=CN,
    dt_max=0.5,
)

n_stop = 200
dt_init = 1e-6
n_init = 5
run(simulation, n_stop=n_stop, dt_init=dt_init, n_init=n_init)

c = simulation['c']

In [ ]:
Ly = 1.0
line_start = (0.0, 0.0)
line_end = (Ly / np.tan(vartheta), Ly)
points = (line_start, line_end)

time_indices = as_index(c.time_series, (0, 0.25, 0.5, -1), fraction=True)
for i in time_indices:
    fig, ax = plot_colormap(c.series[i], title=f'$c(t={c.time_series[i]:.2f})$')
    ax.plot(
        [p[0] for p in points], [p[1] for p in points],
        color='cyan',
        linestyle='dashed',
    )
    save_figure(f'{c.name}(t={c.time_series[i]:.2f})', thumbnail=(i == -1))(fig)

In [ ]:
time_slice = slice(0, None, 2)
titles = [f'${c.name}(t={t:.3f})$' for t in c.time_series[time_slice]]

anim = create_animation(
    plot_colormap,
    colorbar=False,
)(c.series[time_slice], title=titles)
anim_path = save_figure(f'{c.name}(t)', return_path=True)(anim)

display_animation(anim_path)